# Validation Module — Workflow Demo

This notebook demonstrates the four validators in `validation_metrics.py`.
All operate on pre-computed JSON files in `results_analysis/` — no model weights or GPU required.

| Class | Purpose |
|---|---|
| `BootstrapAUROCValidator` | Stratified bootstrap CIs on AUROC and AP (real crops only) |
| `LooDetectionAnalyzer` | Leave-one-out stability on Bełchatów detection record |
| `LeakageAuditor` | Three-check data-leakage and independence audit |
| `HeldOutEvaluator` | Per-site evaluation on held-out and train-as-negative sites |

**Prerequisites:** `numpy` only (matplotlib optional for the LOO plot).

In [ ]:
import sys
from pathlib import Path

# Resolve repo root regardless of working directory
repo_root = Path.cwd()
while repo_root.name not in ('methane-api', '') and repo_root != repo_root.parent:
    repo_root = repo_root.parent
scripts_dir = repo_root / 'scripts'
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

print(f'Repo root: {repo_root}')

In [ ]:
from validation.validation_metrics import (
    BootstrapAUROCValidator,
    LooDetectionAnalyzer,
    LeakageAuditor,
    HeldOutEvaluator,
)

print('Imports OK')

---
## Part 1 — Bootstrap AUROC / Average Precision

Stratified bootstrap on the 14 real-positive + 22 real-negative crops only
(synthetics excluded, addressing Reviewer 2).

In [ ]:
boot_validator = BootstrapAUROCValidator(n_bootstrap=2000, seed=42)
boot_results   = boot_validator.Run()

boot_validator.PrintSummary(boot_results)

In [ ]:
# Inspect the bootstrap distribution percentiles
d = boot_results['distribution']
print(f"AUROC percentiles: p5={d['auroc_p5']:.4f}  p25={d['auroc_p25']:.4f}  "
      f"p50={d['auroc_p50']:.4f}  p75={d['auroc_p75']:.4f}  p95={d['auroc_p95']:.4f}")
print(f"AP    percentiles: p5={d['ap_p5']:.4f}  p25={d['ap_p25']:.4f}  "
      f"p50={d['ap_p50']:.4f}  p75={d['ap_p75']:.4f}  p95={d['ap_p95']:.4f}")

In [ ]:
# What if we want a custom number of bootstrap resamples?
quick_boot = BootstrapAUROCValidator(n_bootstrap=500, seed=0)
qr = quick_boot.Run()
print(f"Quick (n=500): AUROC={qr['auroc']['point']:.4f}  "
      f"90% CI=[{qr['auroc']['ci_90_lo']:.4f}, {qr['auroc']['ci_90_hi']:.4f}]")

In [ ]:
boot_validator.SaveResults(boot_results)
print('Saved.')

---
## Part 2 — LOO Detection Stability

Leave-one-out analysis on the Bełchatów timeseries.  Two passes:
- **Pass A** — hold out one scene at a time
- **Pass B** — hold out all scenes from one calendar month at a time

In [ ]:
loo = LooDetectionAnalyzer()
loo_results = loo.Run(loo_pass='both')

loo.PrintSummary(loo_results)

In [ ]:
# Inspect the most influential scene
pass_a = loo_results['pass_a_scene_level']
most_infl = max(pass_a, key=lambda r: r['influence_detect_rate'] or 0)

print(f"Most influential scene  : {most_infl['held_out_month']}")
print(f"  sc_cfar               : {most_infl['held_out_sc_cfar']:.2f}")
print(f"  Influence (det. rate) : {most_infl['influence_detect_rate']:.4f}")
print(f"  Influence (mean sc)   : {most_infl['influence_mean_sc']:.4f}")
print(f"\nVerdict: {loo_results['stability_verdict']}")

In [ ]:
# Optional: generate the influence plot (requires matplotlib)
try:
    import matplotlib  # noqa
    loo.MakePlot(loo_results)
    print(f'Plot saved to {loo._plot_path}')
except ImportError:
    print('matplotlib not available — skipping plot.')

loo.SaveResults(loo_results)
print('Saved.')

---
## Part 3 — Leakage Audit

Three checks:
1. Site overlap with training set
2. Temporal proximity between training and evaluation acquisitions
3. Conformal calibration set spatial independence (50 km exclusion)

In [ ]:
auditor = LeakageAuditor()
audit_results = auditor.Run()

auditor.PrintSummary(audit_results)

In [ ]:
# Inspect site overlap in detail
print(f"Training crop count: {audit_results['training_crop_count']}\n")
print(f"{'Site':<14} {'In training':>12} {'Roles':<40} {'N crops':>8}")
print('-' * 78)
for site, info in audit_results['candidate_site_overlap'].items():
    roles = ', '.join(info['roles']) if info['roles'] else '—'
    print(f"{site:<14} {str(info['in_training']):>12} {roles:<40} {info['n_crops']:>8}")

In [ ]:
# Check temporal proximity flags
n_flags = sum(
    1 for fs in audit_results['temporal_proximity'].values()
    for f in fs if f['flag'] != 'OK'
)
print(f'Temporal leakage flags: {n_flags}')

cal = audit_results['conformal_calibration_independence']
print(f"Calibration sites within 50 km of a candidate: "
      f"{len(cal['sites_within_exclusion_radius'])} / {cal['checked_n']} checked")
if not cal['sites_within_exclusion_radius']:
    print('\n✓  No conformal calibration contamination detected.')

In [ ]:
auditor.SaveResults(audit_results)
print('Saved.')

---
## Part 4 — Held-out Evaluation

Per-site v8 performance on:
- **Truly held-out** sites (never seen in training)
- **Trained as negative** sites (positive detection overrides the training label)

In [ ]:
evaluator = HeldOutEvaluator()
eval_results = evaluator.Run()

evaluator.PrintSummary(eval_results)

In [ ]:
# Inspect per-acquisition detail for Bełchatów
site = 'belchatow'
if site in eval_results['sites']:
    o = eval_results['sites'][site]
    print(f"{site.title()} — {o['category_label']}")
    print(f"{'Date':<14} {'S/C':>8} {'Above τ':>10} {'CFAR':>8}")
    print('-' * 45)
    for a in o['acquisitions']:
        print(f"{str(a['date']):<14} {a['sc_ratio']:>8.2f} "
              f"{'✓' if a['above_conformal_tau'] else '·':>10} "
              f"{'✓' if a['cfar_detect'] else '·':>8}")

In [ ]:
evaluator.SaveResults(eval_results)
print('Saved.')